# Independent migration verification

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
from pathlib import Path
import csv,json,pickle,hashlib,sys,math
import numpy as np
from contract import ROOT as REPO,REFERENCE,CODE,PROV,read_json,write_json,sha,verify_inputs
PHASE=REFERENCE
ROOT=REPO/'outputs/analysis'
ROOT.mkdir(parents=True,exist_ok=True)
def csv_out(path,rows):
    with path.open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
def csvread(path):
    with Path(path).open() as f:return list(csv.DictReader(f))

META=REPO/'evidence/baselines/solid_rally_development'
TRACE=REPO/'evidence/traces/solid_rally_development'
def compare_csv(actual,expected):
    a,b=csvread(actual),csvread(expected)
    assert len(a)==len(b),(actual,len(a),len(b))
    for i,(x,y) in enumerate(zip(a,b)):
        assert x.keys()==y.keys(),(actual,'columns')
        for key in x:
            try:
                xv,yv=float(x[key]),float(y[key])
                assert math.isclose(xv,yv,abs_tol=1e-6,rel_tol=1e-5),(actual,i,key,xv,yv)
            except ValueError:assert x[key]==y[key],(actual,i,key,x[key],y[key])
    return len(a)
def score_parity():
    from preference_scoring import load_preference_model
    model=load_preference_model()
    export=REPO/'outputs/exports/solid_rally_development/valid_window_trajectories.jsonl'
    rows=[json.loads(line) for line in export.open()]
    features=np.asarray([r['pair_input_L_then_R'] for r in rows]);expected=np.asarray([r['q_increase'] for r in rows])
    actual=model.predict_q(features)
    # Independent affine/logit calculation; does not call predict_proba.
    independent=np.mean([1/(1+np.exp(np.clip(features*s.scale_+s.min_,0,1)@m.coef_[0]+float(m.intercept_[0]))) for m,s in zip(model.models,model.scalers)],axis=0)
    np.testing.assert_allclose(actual,expected,atol=1e-6,rtol=1e-5)
    np.testing.assert_allclose(independent,expected,atol=1e-6,rtol=1e-5)
    result={'pairs':len(rows),'mismatches':int(np.count_nonzero(~np.isclose(actual,expected,atol=1e-6,rtol=1e-5))),'maximum_abs_error':float(np.max(np.abs(actual-expected))),'independent_maximum_abs_error':float(np.max(np.abs(independent-expected)))}
    assert len(rows)==7824
    write_json(ROOT/'score_parity.json',result);return result

def verify_checkpoint(attempt):
    from stable_baselines3 import PPO
    from policy_training import fingerprint
    directory=attempt/'train';cfg=read_json(attempt/'resolved_config.json');result=read_json(directory/'result.json')
    checkpoint=directory/f"checkpoint_{cfg['training_decisions']}.zip"
    assert sha(checkpoint)==result['final_checkpoint_sha256']
    model=PPO.load(checkpoint,device='cpu')
    assert fingerprint(model.policy.state_dict())==result['final_policy_state_sha256']
    assert fingerprint(model.policy.optimizer.state_dict())==result['final_optimizer_state_sha256']
    obs=np.load(directory/'sample_observations.npy');expected=np.load(directory/'expected_actions.npy')
    actions,_=model.predict(obs,deterministic=True);np.testing.assert_array_equal(actions,expected)
    updates=read_json(directory/'updates.json');assert len(updates)==cfg['training_decisions']//2048
    assert all(u['changed_parameter_tensors']>0 for u in updates)
    saved=[u for u in updates if u.get('checkpoint')]
    assert len(saved)==(2 if cfg['training_decisions']==4096 else 5)
    for u in saved:assert sha(directory/Path(u['checkpoint']).name)==u['checkpoint_sha256']
    return {'checkpoint':str(checkpoint.relative_to(REPO)),'decisions':cfg['training_decisions'],'rollouts':len(updates),'actions_reloaded':len(actions),'sha256':sha(checkpoint)}

def verify_accounting():
    results=[]
    for run,attempt,budget in [('P1-TRAIN-SMOKE','attempt-02',4096),('P1-TASK','attempt-01',51200),('P1-MAX','attempt-01',51200)]:
        directory=REFERENCE/'runs'/run/attempt
        cfg=read_json(directory/'resolved_config.json');result=read_json(directory/'train/result.json')
        episodes=read_json(directory/'train/episode_summaries.json')
        assert sum(e['decisions'] for e in episodes)==budget
        assert result['actual_training_decisions']==budget
        assert len(read_json(directory/'train/updates.json'))==budget//2048
        evals=[read_json(directory/f'evaluate-{s}/result.json') for s in cfg['evaluation']['requested_simulator_initialization_seeds']]
        assert all(e['status']=='PASS' and e['evaluation_decisions']==600 for e in evals)
        results.append({'run':run,'training_decisions':budget,'rollouts':budget//2048,'evaluations':len(evals),'task_events':sum(e['task_return'] for e in episodes)})
    failed=read_json(REFERENCE/'runs/P1-TRAIN-SMOKE/attempt-01/result.json')
    assert failed['status']=='FAIL'
    return {'successful_runs':results,'failed_attempt_preserved':True,'TASK_training_OS_exit_code':'unavailable in historical outer supervisor; artifacts verified separately'}

def verify_timing():
    episodes=csvread(ROOT/'evaluation_episodes.csv')
    assert len(episodes)==30
    for r in episodes:
        assert int(r['decisions'])==600 and int(r['valid_pairs'])==39 and float(r['pair_coverage'])==1
        assert r['terminated']=='False' and r['truncated']=='True'
        directory=REFERENCE/Path(r['events_path']).parent
        result=read_json(directory/'result.json')
        assert result['runtime']['initialization_handshake_seeds']==[int(r['requested_Unity_seed'])]
    unique={c:len({r['trajectory_sha256'] for r in episodes if r['condition']==c}) for c in ['TASK','MAX','RANDOM']}
    assert unique=={'TASK':1,'MAX':1,'RANDOM':10}
    return {'decisions':18000,'pairs':1170,'first_pair_seconds':6,'window_seconds':3,'distinct_trajectories':unique,'per_reset_Unity_reseed':False,'independent_Unity_seed_diversity':'not demonstrated'}

def verify_parity():
    result={'scoring':score_parity(),'tables':{}}
    for name in ['baseline_results.csv','evaluation_episodes.csv','segment_review.csv','training_episode_series.csv','valid_pairs.csv']:
        result['tables'][name]=compare_csv(ROOT/name,META/name)
    out=REPO/'outputs/exports/solid_rally_development'
    for name in ['trace_index.csv','support_summary.csv']:result['tables'][name]=compare_csv(out/name,TRACE/name)
    assert read_json(out/'source_manifest.json')==read_json(TRACE/'source_manifest.json')
    def numbers(x,y):
        if isinstance(x,dict):
            assert x.keys()==y.keys()
            for k in x:numbers(x[k],y[k])
        elif isinstance(x,list):
            assert len(x)==len(y)
            for a,b in zip(x,y):numbers(a,b)
        elif isinstance(x,(int,float)):assert math.isclose(x,y,abs_tol=1e-6,rel_tol=1e-5)
        else:assert x==y,(x,y)
    numbers(read_json(ROOT/'compute_estimate.json'),read_json(META/'compute_estimate.json'))
    result['source_files']=32;result['compute_estimate']='all fields match';result['status']='PASS'
    write_json(ROOT/'migration_verification.json',result);return result
print('Independent migration verification definitions/execution completed.')


Frozen runtime contract definitions/execution completed.
Independent migration verification definitions/execution completed.
